# Cross-Validation

**Goal:** evaluate a model reliably — not just on one lucky/unlucky split.

A single `train_test_split` gives one number. Change `random_state` and the score changes too. So how do we know if the model is really good?

**Solution:** k-fold cross-validation — split into k pieces, train k times, average the score.

---

## Pipeline of this notebook

1. Generate synthetic regression data (self-contained — no CSV)
2. Show that a single split's score is unreliable — varies with `random_state`
3. Use k-fold CV to get a stable estimate
4. Use `GridSearchCV` to find the best hyperparameter using CV

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score

np.random.seed(42)

## Step 1 — Generate Self-Contained Data

`make_regression` builds a regression dataset where the true relationship is known. Here:
- 200 samples
- 10 features
- some noise added

In [3]:
X, y = make_regression(n_samples=200, n_features=10, noise=15, random_state=42)
print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (200, 10)
y shape: (200,)


## Step 2 — The Problem with a Single Split

Let's fit the same model with different `random_state` values for `train_test_split` and see how much the R² score varies.

In [4]:
scores_single = []
for rs in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=rs)
    model = LinearRegression().fit(X_train, y_train)
    score = model.score(X_test, y_test)
    scores_single.append(score)

df = pd.DataFrame({'random_state': range(10), 'R2_score': scores_single})
print(df.round(3))
print(f'\nMin: {min(scores_single):.3f}   Max: {max(scores_single):.3f}   Spread: {max(scores_single) - min(scores_single):.3f}')

   random_state  R2_score
0             0     0.992
1             1     0.992
2             2     0.993
3             3     0.990
4             4     0.991
5             5     0.993
6             6     0.992
7             7     0.991
8             8     0.993
9             9     0.992

Min: 0.990   Max: 0.993   Spread: 0.004


**Observation:** the same model, same data — but R² varies depending on which random split you got. You cannot trust a single split.

---

## Step 3 — k-Fold Cross-Validation

Split the data into k=5 folds. Train 5 times — each time using 4 folds for training and 1 fold for testing. Average the 5 scores.

```
Fold 1:  [TEST][train][train][train][train]
Fold 2:  [train][TEST][train][train][train]
Fold 3:  [train][train][TEST][train][train]
Fold 4:  [train][train][train][TEST][train]
Fold 5:  [train][train][train][train][TEST]
```

Every observation gets used for testing exactly once.

In [5]:
model = LinearRegression()
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print('Individual fold scores:', np.round(cv_scores, 3))
print(f'Mean R2: {cv_scores.mean():.3f}')
print(f'Std  R2: {cv_scores.std():.3f}')

Individual fold scores: [0.995 0.994 0.989 0.991 0.989]
Mean R2: 0.991
Std  R2: 0.002


**The mean is the reliable estimate.** Standard deviation tells you how stable the model is across folds — a low std means the model behaves consistently.

---

## Step 4 — GridSearchCV: Find the Best Hyperparameter

Now apply CV to **hyperparameter tuning**. We try Ridge regression with different values of `alpha` (regularisation strength) and pick the best — using cross-validation to evaluate each.

`GridSearchCV` does this automatically:
1. Try each candidate value
2. For each, run k-fold CV
3. Return the one with the best average CV score

In [ ]:
param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}

grid = GridSearchCV(Ridge(), param_grid, cv=5, scoring='r2')
grid.fit(X, y)

results = pd.DataFrame({
    'alpha': param_grid['alpha'],
    'mean_R2': grid.cv_results_['mean_test_score'].round(3),
    'std_R2':  grid.cv_results_['std_test_score'].round(3)
})
print(results)

print(f'\nBest alpha: {grid.best_params_["alpha"]}')
print(f'Best mean R2: {grid.best_score_:.3f}')

GridSearchCV picks the alpha that gives the highest average CV R² — a much more trustworthy choice than picking based on a single split.

---

## Summary

| Approach | What you get |
|----------|------------|
| Single split | One number — unreliable, depends on random_state |
| k-fold CV | Average of k scores — stable estimate |
| GridSearchCV | Best hyperparameter chosen via k-fold CV |

> Use CV when you want reliable evaluation. Use GridSearchCV when you also want to tune hyperparameters.